In [17]:
import cv2
from ultralytics import YOLO

In [18]:
# Load Stage 1: Person/Half-Person Tracking Model
person_detection_model = YOLO(model="/home/ai_vison/Desktop/PPE/model_finetune/person/runs/2026-06-04 16:54:10 150 l/weights/best.pt")

# Load Stage 2: PPE Detection Model (Path from your pipeline.ipynb)
ppe_detection_model = YOLO(model="/home/ai_vison/Desktop/PPE/model_finetune/ppe_detect/runs/2026-06-02 18:26:19 200 l/weights/best.pt")

In [19]:
# 2. Open Video Stream (Replace with 0 for live webcam/robot camera)
video_path = "./inf_vid/Construction Safety Training clip 01.mp4"
cap = cv2.VideoCapture(video_path)

In [20]:
# Verify video opened successfully
if not cap.isOpened():
    print("Error: Could not open video source.")
    exit()

In [21]:
# --- NEW: Get the video's original FPS and calculate the required delay ---
video_fps = cap.get(cv2.CAP_PROP_FPS)
# Default to 30 if metadata is missing, otherwise calculate delay in ms
frame_delay = int(1000 / video_fps) if video_fps > 0 else 33
print(f"Video FPS: {video_fps} | Calculated Delay: {frame_delay}ms")

Video FPS: 24.013261958211018 | Calculated Delay: 41ms


In [22]:
# --- NEW: Set up video writer to save output ---
output_path = "./inf_vid/output_ppe_detected.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # MP4 codec

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(output_path, fourcc, video_fps, (frame_width, frame_height))

if not out.isOpened():
    print("Error: Could not create video writer. Trying XVID codec instead...")
    fourcc = cv2.VideoWriter_fourcc(*'XVID')  # Fallback codec
    out = cv2.VideoWriter(output_path.replace('.mp4', '.avi'), fourcc, video_fps, (frame_width, frame_height))

print(f"Video will be saved to: {output_path}")
print(f"VideoWriter status: {'OPEN' if out.isOpened() else 'FAILED'}")

Video will be saved to: ./inf_vid/output_ppe_detected.mp4
VideoWriter status: OPEN


In [23]:
import time

while cap.isOpened():
    start_time = time.time()  # Track how long processing takes
    
    ret, frame = cap.read()
    if not ret:
        print("Video stream finished.")
        break

    # --- STAGE 1: Run YOLO Tracking ---
    results = person_detection_model.track(frame, persist=True, classes=[0, 1], conf=0.1, iou=0.7, verbose=False)

    if results[0].boxes is not None and len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
        
        track_ids = (
            results[0].boxes.id.cpu().numpy().astype(int) 
            if results[0].boxes.id is not None 
            else range(len(boxes))
        )

        for box, track_id in zip(boxes, track_ids):
            x1, y1, x2, y2 = box
            
            # Safeguard against out-of-bounds coordinates
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
            
            # Draw the human bounding box (Blue)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(frame, f"Person ID: {track_id}", (x1, y1 - 10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

            # --- STAGE 2: PPE Detection ---
            # 1. Crop the human from the original frame
            person_crop = frame[y1:y2, x1:x2]
            
            # Prevent passing empty arrays if bounding box logic fails
            if person_crop.size == 0:
                continue

            # 2. Run the PPE model on the cropped image
            # Using imgsz=[480, 192] to match the optimal shape found in pipeline.ipynb
            ppe_results = ppe_detection_model.predict(
                source=person_crop, 
                imgsz=[480, 192], 
                classes=[0, 1, 2], 
                conf=0.5, 
                verbose=False
            )

            # 3. Process PPE results and draw them on the main frame
            if ppe_results[0].boxes is not None and len(ppe_results[0].boxes) > 0:
                ppe_boxes = ppe_results[0].boxes.xyxy.cpu().numpy().astype(int)
                ppe_clss = ppe_results[0].boxes.cls.cpu().numpy().astype(int)

                for ppe_box, cls_id in zip(ppe_boxes, ppe_clss):
                    px1, py1, px2, py2 = ppe_box

                    # Remap cropped coordinates back to the original full frame
                    master_x1 = x1 + px1
                    master_y1 = y1 + py1
                    master_x2 = x1 + px2
                    master_y2 = y1 + py2

                    # Get class name (boot, helmet, jacket)
                    label = ppe_detection_model.names[cls_id]

                    # Draw PPE bounding box (Green)
                    cv2.rectangle(frame, (master_x1, master_y1), (master_x2, master_y2), (0, 255, 0), 2)
                    cv2.putText(frame, label, (master_x1, master_y1 - 10), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.imshow("Stage 1 & 2 - Real-time PPE Tracking", frame)
    # --- Write the annotated frame to the output video ---
    out.write(frame)

    # --- Dynamically calculate wait time ---
    elapsed_time = int((time.time() - start_time) * 1000)
    actual_delay = max(1, frame_delay - elapsed_time)

    if cv2.waitKey(actual_delay) & 0xFF == ord('q'):
        break

Video stream finished.


In [24]:
# Clean up and release system resources
out.release()  # NEW: Release the video writer
cap.release()
cv2.destroyAllWindows()